## Load the cleaned dataset

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Find the project root folder
PROJECT_ROOT = Path.cwd()

# If the notebook is opened from the notebooks folder,
# move up one level to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Path to the cleaned dataset
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "paysim_cleaned.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

# Load the cleaned dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)


## Inspect columns and data types

In [ ]:

# Display all column names
print("COLUMN NAMES")
print("=" * 50)

for column in df.columns:
    print(column)

# Display data types and non-null counts
print("\nDATASET INFORMATION")
print("=" * 50)

df.info()

# Display the first five transactions
print("\nFIRST FIVE TRANSACTIONS")
print("=" * 50)

display(df.head())


## Analyze fraud vs. legitimate transactions

In [ ]:

# Count legitimate and fraudulent transactions
fraud_counts = df["isFraud"].value_counts().sort_index()

print("TRANSACTION CLASS DISTRIBUTION")
print("=" * 45)
print(f"Legitimate transactions (0): {fraud_counts.get(0, 0):,}")
print(f"Fraudulent transactions (1): {fraud_counts.get(1, 0):,}")

# Calculate percentages
fraud_percentages = (
    df["isFraud"].value_counts(normalize=True).sort_index() * 100
)

print("\nTRANSACTION PERCENTAGES")
print("=" * 45)
print(f"Legitimate: {fraud_percentages.get(0, 0):.4f}%")
print(f"Fraudulent: {fraud_percentages.get(1, 0):.4f}%")

# Plot transaction distribution
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="isFraud"
)

plt.title("Legitimate vs Fraudulent Transactions")
plt.xlabel("Transaction Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.xticks([0, 1], ["Legitimate", "Fraud"])
plt.tight_layout()
plt.show()


## Analyze fraud by transaction type

In [ ]:

# Count transactions by type and fraud status
print("TRANSACTION COUNTS BY TYPE")
print("=" * 55)

transaction_counts = pd.crosstab(
    df["type"],
    df["isFraud"]
)

transaction_counts.columns = [
    "Legitimate",
    "Fraudulent"
]

print(transaction_counts)

# Calculate fraud percentage within each transaction type
fraud_by_type = (
    df.groupby("type")["isFraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("mean", ascending=False)
)

fraud_by_type.columns = [
    "Total Transactions",
    "Fraudulent Transactions",
    "Fraud Rate"
]

fraud_by_type["Fraud Rate (%)"] = (
    fraud_by_type["Fraud Rate"] * 100
)

print("\nFRAUD RATE BY TRANSACTION TYPE")
print("=" * 55)

display(
    fraud_by_type[
        [
            "Total Transactions",
            "Fraudulent Transactions",
            "Fraud Rate (%)"
        ]
    ].round(4)
)

# Visualize fraudulent transaction counts
plt.figure(figsize=(9, 5))

sns.barplot(
    data=fraud_by_type.reset_index(),
    x="type",
    y="Fraudulent Transactions"
)

plt.title("Fraudulent Transactions by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Fraudulent Transactions")
plt.tight_layout()
plt.show()


### Analyze transaction amounts

In [ ]:

# Separate legitimate and fraudulent transaction amounts
legitimate_amounts = df.loc[df["isFraud"] == 0, "amount"]
fraud_amounts = df.loc[df["isFraud"] == 1, "amount"]

print("TRANSACTION AMOUNT SUMMARY")
print("=" * 45)

print("\nLegitimate transactions:")
print(legitimate_amounts.describe())

print("\nFraudulent transactions:")
print(fraud_amounts.describe())

# Compare amount distributions using a box plot.
# A log scale helps show a wide range of transaction amounts.
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="isFraud",
    y="amount",
    showfliers=False
)

plt.yscale("log")
plt.title("Transaction Amounts: Legitimate vs Fraud")
plt.xlabel("Transaction Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Amount (log scale)")
plt.xticks([0, 1], ["Legitimate", "Fraud"])
plt.tight_layout()
plt.show()


## Analyze account balance changes

In [ ]:

# Calculate balance changes for the originating account
df["origin_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

# Calculate balance changes for the destination account
df["destination_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

# Compare average balance changes
balance_summary = df.groupby("isFraud")[
    [
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "origin_balance_change",
        "destination_balance_change"
    ]
].mean()

balance_summary.index = ["Legitimate", "Fraudulent"]

print("AVERAGE ACCOUNT BALANCE COMPARISON")
print("=" * 55)

display(balance_summary.round(2))

# Visualize the originating account's balance change
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="isFraud",
    y="origin_balance_change",
    showfliers=False
)

plt.title("Originating Account Balance Change")
plt.xlabel("Transaction Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Balance Before - Balance After")
plt.xticks([0, 1], ["Legitimate", "Fraud"])
plt.tight_layout()
plt.show()


## Check the complete balance summary

In [ ]:

# Display the complete balance summary
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(balance_summary.round(2))

# Check whether any calculated values are missing
print("\nMissing values in balance summary:")
print(balance_summary.isna().sum())

# Show the average balance changes only
print("\nAverage balance changes:")
print(
    df.groupby("isFraud")[
        ["origin_balance_change", "destination_balance_change"]
    ].mean().round(2)
)


## Analyze the existing fraud flag

In [ ]:

# Count existing fraud flags
print("EXISTING FRAUD FLAG ANALYSIS")
print("=" * 45)

print("\nFlag counts:")
print(df["isFlaggedFraud"].value_counts().sort_index())

print("\nActual fraud vs existing flag:")
display(
    pd.crosstab(
        df["isFraud"],
        df["isFlaggedFraud"],
        rownames=["Actual Fraud (isFraud)"],
        colnames=["Existing Flag (isFlaggedFraud)"]
    )
)

# Count actual fraud cases that were flagged
actual_fraud = df["isFraud"] == 1
flagged = df["isFlaggedFraud"] == 1

print("\nActual fraud transactions:", int(actual_fraud.sum()))
print("Transactions flagged by existing system:", int(flagged.sum()))
print(
    "Actual fraud transactions flagged:",
    int((actual_fraud & flagged).sum())
)


## Save EDA findings

In [ ]:

# Summarize key EDA findings
total_transactions = len(df)
fraud_count = int(df["isFraud"].sum())
legitimate_count = total_transactions - fraud_count
fraud_percentage = fraud_count / total_transactions * 100

flagged_count = int(df["isFlaggedFraud"].sum())
fraud_flagged_count = int(
    ((df["isFraud"] == 1) & (df["isFlaggedFraud"] == 1)).sum()
)

eda_summary = {
    "Total transactions": total_transactions,
    "Legitimate transactions": legitimate_count,
    "Fraudulent transactions": fraud_count,
    "Fraud percentage": round(fraud_percentage, 4),
    "Transaction types": df["type"].nunique(),
    "Duplicate rows": int(df.duplicated().sum()),
    "Missing values": int(df.isna().sum().sum()),
    "Transactions flagged by existing system": flagged_count,
    "Actual fraud transactions flagged": fraud_flagged_count,
}

print("EDA SUMMARY")
print("=" * 50)

for metric, value in eda_summary.items():
    print(f"{metric}: {value:,}" if isinstance(value, int) else f"{metric}: {value}")


## Test feature engineering

In [ ]:

from pathlib import Path
import sys

# Add the project root to Python's import path
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from ml.features.feature_engineering import create_features

# Load just 10 rows for testing
sample_df = pd.read_csv(DATA_PATH, nrows=10)

print("Original sample shape:", sample_df.shape)

# Apply feature engineering
sample_features = create_features(sample_df)

print("Shape after feature engineering:", sample_features.shape)

# Display original and new features
new_columns = [
    "amount_to_origin_balance_ratio",
    "amount_to_destination_balance_ratio",
    "origin_balance_error",
    "destination_balance_error"
]

display(
    sample_features[
        [
            "type",
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "oldbalanceDest",
            "newbalanceDest",
            "isFraud"
        ] + new_columns
    ]
)

# Basic checks
assert len(sample_features) == len(sample_df)
assert all(column in sample_features.columns for column in new_columns)
assert "isFraud" in sample_features.columns

print("Feature engineering sample test: PASSED")


## Verify the feature-engineered dataset

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURES_PATH = (
    PROJECT_ROOT / "data" / "processed" / "paysim_features.csv"
)

# Read only a small sample for verification
features_sample = pd.read_csv(FEATURES_PATH, nrows=1000)

print("FEATURE DATASET VERIFICATION")
print("=" * 50)

print("File exists:", FEATURES_PATH.exists())
print("Full dataset rows:", 6_362_620)
print("Sample rows loaded:", len(features_sample))
print("Number of columns:", len(features_sample.columns))

print("\nColumn names:")
print(features_sample.columns.tolist())

print("\nMissing values in sample:")
print(features_sample.isna().sum())

print("\nTarget distribution in sample:")
print(features_sample["isFraud"].value_counts())

print("\nVerification completed.")
